### Supplement: 2-Structured Outputs with Pydantic Deep Dive

This supplement notebook breaks down `2-structured.py` cell by cell. It covers **Structured Outputs** using Pydantic and the OpenAI Python SDK:

1. **What is Pydantic and `BaseModel`?**: Defining data schemas with strict types.
2. **Schema Generation**: How `CalendarEvent` is converted to a JSON Schema under the hood.
3. **`create` vs. `parse`**: Understanding `client.beta.chat.completions.parse(...)`.
4. **`message.content` vs. `message.parsed`**:
   - `message.content`: The raw JSON string returned by the LLM.
   - `message.parsed`: The deserialized, validated Pydantic object!
5. **Accessing Typed Attributes & Converting to Python Dicts**: Dot-notation vs. `.model_dump()`.

---

##### Architectural Flow:
```
1. Define Pydantic Schema:
   class CalendarEvent(BaseModel):
       name: str
       date: str
       participants: list[str]
            │
            ▼
2. SDK converts schema to JSON Schema -> sent to OpenAI API in request body
            │
            ▼
3. OpenAI API uses Constrained Sampling (guarantees 100% adherence to schema)
            │
            ▼
4. Response arrives back at SDK:
   ├── message.content -> raw JSON string: '{"name": "...", "date": "...", ...}'
   └── message.parsed  -> CalendarEvent(name='...', date='...', participants=[...])
```


#### 1. Imports and Environment Setup

Notice we import:
- `OpenAI`: The API client class.
- `BaseModel`, `Field`: From `pydantic`. `BaseModel` is the base class for defining data contracts and schemas.

In [1]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized successfully.")
print("Pydantic BaseModel class:", BaseModel)


Client initialized successfully.
Pydantic BaseModel class: <class 'pydantic.main.BaseModel'>


#### 2. Defining the Schema Class: `CalendarEvent(BaseModel)`

##### What is `BaseModel`?
`BaseModel` is Pydantic's core class. When you subclass `BaseModel`:
- It parses and validates data according to type hints (`str`, `list[str]`, etc.).
- It can automatically export a standard JSON Schema via `.model_json_schema()`.
- It allows serialization back to dicts via `.model_dump()`.

Let's inspect the class and the JSON Schema sent to OpenAI:

In [2]:
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")

print("Class name:", CalendarEvent.__name__)
print("Inherits from:", [b.__name__ for b in CalendarEvent.__bases__])
print("Declared fields:", list(CalendarEvent.model_fields.keys()))

print("\n--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---")
schema = CalendarEvent.model_json_schema()
print(json.dumps(schema, indent=2))


Class name: CalendarEvent
Inherits from: ['BaseModel']
Declared fields: ['name', 'date', 'participants']

--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


#### 3. The API Call: `client.beta.chat.completions.parse(...)`

##### Why `parse(...)` instead of `create(...)`?
- **`create(...)`**: You get back standard `ChatCompletion`. If you asked for JSON, `message.content` is just a string, and you would have to manually call `json.loads(response)` and hope the model didn't hallucinate invalid syntax.
- **`parse(...)`**: A high-level helper in the OpenAI SDK that:
  1. Sends the Pydantic schema to OpenAI with `strict: True`.
  2. The model generates strictly conforming JSON tokens via constrained decoding.
  3. The SDK automatically validates the JSON and instantiates your `CalendarEvent` class!
  4. The instantiated object is placed in `completion.choices[0].message.parsed`!

In [3]:
prompt_messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

completion = client.beta.chat.completions.parse(
    model="gpt-5-nano",
    messages=prompt_messages,
    response_format=CalendarEvent,
)

print("Parsed completion call successful!")
print("Completion object type:", type(completion))
print("Choices length:", len(completion.choices))


Parsed completion call successful!
Completion object type: <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[CalendarEvent]'>
Choices length: 1


#### 4. Comparing `message.content` vs. `message.parsed`

This is the most critical distinction in Structured Outputs:
1. **`message.content`**: Contains the **raw JSON string** sent across the wire by the LLM.
2. **`message.parsed`**: Contains the **instantiated Python Pydantic object** (`CalendarEvent`).

Let's inspect both with `type()` and `repr()`:

In [4]:
message = completion.choices[0].message

print("=== 1. message.content (Raw Wire JSON) ===")
print("Type:", type(message.content))
print("Raw string value:", repr(message.content))

print("\n=== 2. message.parsed (Deserialized Pydantic Object) ===")
print("Type:", type(message.parsed))
print("Is instance of CalendarEvent?:", isinstance(message.parsed, CalendarEvent))
print("Object representation:", repr(message.parsed))

print("\n=== 3. message.refusal ===")
print("Refusal status:", message.refusal)


=== 1. message.content (Raw Wire JSON) ===
Type: <class 'str'>
Raw string value: '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

=== 2. message.parsed (Deserialized Pydantic Object) ===
Type: <class '__main__.CalendarEvent'>
Is instance of CalendarEvent?: True
Object representation: CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

=== 3. message.refusal ===
Refusal status: None


#### 5. Accessing Typed Attributes & Converting to Python Dict

Because `event` is a `CalendarEvent` object:
- You get typed attribute access (dot-notation) with IDE autocompletion: `event.name`, `event.date`, `event.participants`.
- `event.participants` is a genuine Python `list` of strings!
- You can convert the object to a standard Python dictionary using `event.model_dump()`.

In [5]:
event: CalendarEvent = message.parsed

print("--- Accessing Typed Attributes ---")
print(f"event.name:         {event.name} (type: {type(event.name)})")
print(f"event.date:         {event.date} (type: {type(event.date)})")
print(f"event.participants: {event.participants} (type: {type(event.participants)})")
print(f"First participant:  {event.participants[0]} (type: {type(event.participants[0])})")

print("\n--- Converting to Standard Python Dictionary (.model_dump()) ---")
event_dict = event.model_dump()
print("Type of event_dict:", type(event_dict))
print("Dictionary content:", event_dict)


--- Accessing Typed Attributes ---
event.name:         Science Fair (type: <class 'str'>)
event.date:         Friday (type: <class 'str'>)
event.participants: ['Alice', 'Bob'] (type: <class 'list'>)
First participant:  Alice (type: <class 'str'>)

--- Converting to Standard Python Dictionary (.model_dump()) ---
Type of event_dict: <class 'dict'>
Dictionary content: {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


#### 6. Visualizing the Complete Structured Response Object

Let's dump the entire `completion` object to inspect everything OpenAI returned, including token usage and choice metadata:

In [6]:
full_dict = completion.model_dump()

print("Full Completion Dictionary:")
print(json.dumps(full_dict, indent=2))


Full Completion Dictionary:
{
  "id": "chatcmpl-EOf16vafxtVG3BPNvUUHu3OcyanFN",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789546248,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 478,
    "prompt_tokens": 117,
    "total_tokens": 595,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
   